## Setup
#### Load the API key and relevant Python libaries.

##### this notebook runs in Colab

In [89]:
from collections import defaultdict
import json
import numpy as np
import pandas as pd
import os
import random
import io
from dotenv import dotenv_values
import openai
import os
from copy import deepcopy
import time
import ast
import re
import pprint
from openai import OpenAI

# LLM Open AI

In [70]:
# Get the first key from the uploaded dictionary
env_file_key = "../../auixiliary/env_GENERAL"

# Open the file and read its content
with open(env_file_key, 'r', encoding='utf-8') as file:
    env_content = file.read()

# Load the content into a variable
env_variables = dotenv_values(stream=io.StringIO(env_content))

api_key = env_variables['OPENAI_API_KEY']
# openai.api_key = api_key

client = OpenAI(
    # This is the default and can be omitted
    api_key=api_key,
)

# Models

In [71]:
def chat_gpt(prompt):
    response = client.chat.completions.create(
        model="gpt-5",
        messages=prompt
    )
    return response.choices[0].message.content.strip()

In [72]:
def parse_response(json_block):
    return ast.literal_eval(json_block)

# Iterate and Save Use Riskiness Results

# Functions

In [73]:
def replace_key(d, old_key, new_key):
  """
  Replace `old_key` with `new_key` in dictionary `d`.
  The associated value is retained.
  """
  if old_key in d:
      d[new_key] = d.pop(old_key)
  return d

## Read In Prompt Result

In [74]:
def read_prompt_output(input_data_location):
  
  # Open the JSON file for reading
  with open(input_data_location, 'r') as file:
      # Load JSON data from file
      file_content = json.load(file)

  return file_content

## READ DATA

### ExploreGen uses

In [75]:
uses_of = "NOK"
data_location = "../../data/"
USES = read_prompt_output(data_location + f"TELCO_uses_ExploreGen_def_{uses_of}.json")

In [76]:
i = 1
for el in USES:
    el["ConnectionRating"] = 5
    el["Justification"] = ""
    el = replace_key(el, "AI User", "AI Deployer")
    el["Space"] = ""
    el["IncidentID"] = "ExploreGen-" + str(i)
    el["Link"] = ""
    el["Source"] = "ExploreGen"
    i += 1

In [77]:
USES[0]

{'Use': 1,
 'Domain': 'Biometric identification and categorization of natural persons',
 'Purpose': 'Authenticating callers via voiceprints',
 'Capability': 'Identifying speaker identity from call audio',
 'Space': '',
 'AI Deployer': 'Telecom operator',
 'AI Subject': 'Callers',
 'ConnectionRating': 5,
 'Justification': '',
 'IncidentID': 'ExploreGen-1',
 'Link': '',
 'Source': 'ExploreGen'}

In [78]:
USES[137]

{'Use': 138,
 'Domain': 'Interpersonal Communication',
 'Purpose': 'Transcribing voicemails automatically',
 'Capability': 'Generating text from voice messages',
 'Space': '',
 'AI Deployer': 'Mobile network operator',
 'AI Subject': 'Subscribers',
 'ConnectionRating': 5,
 'Justification': '',
 'IncidentID': 'ExploreGen-138',
 'Link': '',
 'Source': 'ExploreGen'}

### Uses from AIID and Telco Companies

In [79]:
aiid_website_USES = read_prompt_output(data_location + "ai_uses_telco_relevance_equal_or_above_3.json")

In [80]:
aiid_website_USES[0]

{'UseID': 22,
 'IncidentID': 22,
 'ConnectionRating': 3,
 'Justification': "Waze relies on GPS and mobile data networks, which are part of telecommunications infrastructure, to collect and transmit traffic information and route users. While not directly managing telecom networks, it depends on telecom services and affects mobile users' communication and navigation.",
 'Domain': 'Transport and Logistics',
 'Purpose': 'Optimizing route navigation',
 'Capability': 'Forecasting traffic patterns from GPS data',
 'Space': 'Online',
 'AI Deployer': 'Navigation app providers',
 'AI Subject': 'Motorists',
 'Link': 'https://incidentdatabase.ai/cite/22/',
 'Source': 'AIID'}

In [81]:
aiid_website_USES[77]

{'UseID': 1236,
 'IncidentID': 'NVIDIA-6',
 'ConnectionRating': 4,
 'Justification': 'NVIDIA Morpheus enables AI pipelines that detect threats using network telemetry, including DPU-sourced data.',
 'Domain': 'Security and Cybersecurity',
 'Purpose': 'Detecting cyber threats in networks',
 'Capability': 'Classifying threats from telemetry data',
 'Space': 'Physical Restricted / Online',
 'AI Deployer': 'Telecom operators',
 'AI Subject': 'Security teams and subscribers',
 'Link': 'https://developer.nvidia.com/industries/telecommunications',
 'Source': 'Website'}

In [85]:
aiid_website_USES[115]

{'UseID': 1274,
 'IncidentID': 'Cisco-6',
 'ConnectionRating': 4,
 'Justification': 'Cisco describes digital twins and real-time analytics for telcos to improve service resilience and reduce downtime. (Cisco blog on digital resilience)',
 'Domain': 'Management and Operation of critical infrastructure',
 'Purpose': 'Simulating and predicting network behavior',
 'Capability': 'Estimating impact from digital twin models + telemetry',
 'Space': 'Online / Physical Restricted hybrid',
 'AI Deployer': 'Telecom operators / infrastructure providers',
 'AI Subject': 'Network planners / operators',
 'Link': 'https://news-blogs.cisco.com/apjc/2025/03/06/how-telcos-are-reinventing-digital-resilience-with-ai-and-automatio/',
 'Source': 'Website'}

In [90]:
all_uses = USES + aiid_website_USES
len(all_uses)

254

In [105]:
# Calculate 20% of the list size
subset_size = int(len(all_uses) * 0.3)

# Get random subset
random_subset = random.sample(all_uses, subset_size)

In [106]:
random_subset

[{'UseID': 719,
  'IncidentID': 730,
  'ConnectionRating': 4,
  'Justification': 'The incident involves AI-generated personalized voice and video calls delivered over telecommunication networks to voters, directly leveraging telecom infrastructure and services for mass outreach, though the AI itself is not embedded in core telecom operations.',
  'Domain': 'Democracy',
  'Purpose': 'Delivering personalized voter outreach',
  'Capability': 'Generating synthetic voice and video messages',
  'Space': 'Online',
  'AI Deployer': 'Political campaign service providers',
  'AI Subject': 'Voters',
  'Link': 'https://incidentdatabase.ai/cite/730/',
  'Source': 'AIID'},
 {'UseID': 1225,
  'IncidentID': 'Ericsson-5',
  'ConnectionRating': 4,
  'Justification': 'Ericsson embeds explainable AI into rApps for O-RAN, ensuring that network automation decisions can be understood and trusted.',
  'Domain': 'Management and Operation of critical infrastructure',
  'Purpose': 'Making RAN automation explaina

# FINAL PROMPT

In [117]:
# Assuming you have the variables domain, purpose, aiCapability, aiUser, and aiSubject defined with appropriate values

MESSAGES = [
    {
        'role': 'system',
        'content': """You are an experienced AI specialists with regulatory compliance knowledge who works in the field of AI technology and regulation. You are thoughtful, decisive, experienced and conscientious.
        Your understand extremely well different AI use cases based on their descriptions and components. 
        ."""
    },
    {
        'role': 'user',
        'content': """
        You will be provided with a list of about 300 AI uses. Your task is to deduplicate them. To do so, you will need to read through the list and identify any use cases that are similar or identical based on the six component format. 

        Here is the list of all the AI uses:
        "{}"
         
        The key details of each individual AI uses in the list that you should consider are:
            Domain
            Purpose
            Capability
            Space
            AI Deployer
            AI Subject

        Please return the list in the similar format, with the following changes:
            1) If two or more use cases are identical, keep only one of the uses verbatim as is, and remove the duplicates. Store the IncidentID of the removed duplicate uses in the new field: {{"removed_duplicate_uses" : []}}.
            2) If two or more use cases are similar but not identical, group them together and choose one use verbatim as is to represent the whole group that best captures the common elements of the uses. If you cannot find a single such a use, perhaps they should not all be merged, but you might have two or more groups in there. When merging the uses from different sources, those from AIID take precedence over those from Websites, followed by those from ExploreGen. This means that the use from AIID should be kept, followed by the use from Websites, and finally the use from ExploreGen. Those with lower priority should be merged into the main use as similar ones. You can use the following guidelines to determine if two use cases are similar:
                * They have the same or very similar all of the Domain, Purpose, Capability, Space, AI Deployer, and AI Subject.
                * For instance, if AI Deployers or AI Subjects are different, then the uses can be considered different.

                Example of two similar uses that should be merged:

                Use 1:
                    "Domain": "Management and Operation of critical infrastructure",
                    "Purpose": "Automating network incident management",
                    "Capability": "Detecting network issues and dispatching technicians",
                    "Space": "Physical Restricted",
                    "AI_Deployer": "Telecom operators",
                    "AI_Subject": "Telecom network users"
                Use 2:
                    "Domain": "Management and Operation of critical infrastructure",
                    "Purpose": "Proactively resolving network issues",
                    "Capability": "Detecting anomalies and triggering remediation",
                    "Space": "Physical Restricted / Online",
                    "AI Deployer": "Cloud / network service providers",
                    "AI Subject": "Telecom operators"

                Example of two uses that are not sufficiently similar and should NOT be merged:

                Use 1:
                    "Domain": "Management and Operation of critical infrastructure",
                    "Purpose": "Automating network incident management",
                    "Capability": "Detecting network issues and dispatching technicians",
                    "Space": "Physical Restricted",
                    "AI_Deployer": "Telecom operators",
                    "AI_Subject": "Telecom network users"

                Use 2:
                    "Domain": "Crisis Management and Emergency Response",
                    "Purpose": "Rerouting traffic around outages",
                    "Capability": "Acting on damage likelihood predictions",
                    "Space": "Physical Public space",
                    "AI Deployer": "Telecom operator",
                    "AI Subject": "Subscribers"

            The reason is that in this case, the first use is about managing network incidents, while the second use is about responding to crises and emergencies, i.e., they will be used in very different contexts and by different stakeholders. The purposes and subjects are different enough to consider them separate uses.

            3) Add new field "MergedSimilarUses" to all the uses, merged or not. By default, the "MergedSimilarUses" field should be an empty list. When merging similar uses, add the IncidentIDs of the merged uses to the "MergedSimilarUses" field of the resulting use that you selected as representative for the group.
            4) Keep the IncidentID field as is. Renumber UseID sequentially starting from 1.

        The output should be a list of JSON objects, each representing a deduplicated AI use. Each JSON object should contain the following fields:
        - Domain
        - Purpose
        - Capability
        - Space
        - AI Deployer
        - AI Subject
        - MergedSimilarUses

        OUTPUT FORMAT:
        {{
            "removed_duplicate_uses": [list of removed IncidentIDs],
            "deduplicated_uses": [
                {{
                "IncidentID": "original_id",
                "UseID": 1,
                "Domain": "...",
                "Purpose": "...",
                "Capability": "...",
                "Space": "...",
                "AI_Deployer": "...",
                "AI_Subject": "...",
                "MergedSimilarUses": [list of merged IncidentIDs]
                }},
                ... more deduplicated uses
            ]
        }}  

        Ensure to output a *CORRECTLY FORMATTED* LIST OF JSONS!!!
        """
    }
]



def format_prompt(MESSAGES, USE_LIST):
    S = "test {}"
    messages = deepcopy(MESSAGES)
    messages[1]['content'] = messages[1]['content'].format(USE_LIST)
    return messages


# APPLY / RUN

In [118]:
messages = format_prompt(MESSAGES, all_uses)
response = chat_gpt(messages)

In [119]:
RESPONSE = parse_response(response)

In [121]:
RESPONSE

{'removed_duplicate_uses': [],
 'deduplicated_uses': [{'IncidentID': 'ExploreGen-1',
   'UseID': 1,
   'Domain': 'Biometric identification and categorization of natural persons',
   'Purpose': 'Authenticating callers via voiceprints',
   'Capability': 'Identifying speaker identity from call audio',
   'Space': '',
   'AI_Deployer': 'Telecom operator',
   'AI_Subject': 'Callers',
   'MergedSimilarUses': []},
  {'IncidentID': 'ExploreGen-2',
   'UseID': 2,
   'Domain': 'Biometric identification and categorization of natural persons',
   'Purpose': 'Controlling secure facility entry',
   'Capability': 'Recognizing faces from CCTV footage',
   'Space': '',
   'AI_Deployer': 'Telecom operator',
   'AI_Subject': 'Employees',
   'MergedSimilarUses': []},
  {'IncidentID': 'ExploreGen-3',
   'UseID': 3,
   'Domain': 'Biometric identification and categorization of natural persons',
   'Purpose': 'Verifying SIM registration identities',
   'Capability': 'Matching fingerprints from enrollment scan

In [122]:
len(RESPONSE), len(RESPONSE['deduplicated_uses'])

(2, 239)

# SAVE

In [123]:
###############################
# save result
with open(f"../../results/deduplicated_uses_v2.json", "w") as json_file:
    json.dump(RESPONSE, json_file, indent=4)  # 4 spaces of indentation

# THE END